# 03 · Semantic Retrieval with ChromaDB


## Install


In [1]:
%pip install -q chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 417.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/

## Create a collection and add records


In [2]:
import chromadb

client = chromadb.PersistentClient(path="./local_vector_data")
collection = client.get_or_create_collection(name="campus_help_v1")

records = [
    ("library_hours", "The campus library is open from 8 AM to 10 PM on weekdays.", {"topic": "facilities"}),
    ("wifi_help", "Connect to the student Wi-Fi using your campus credentials.", {"topic": "IT"}),
    ("exam_rules", "Bring your student identity card to every examination.", {"topic": "academics"}),
    ("sports_center", "The sports center has badminton courts and a fitness room.", {"topic": "sports"}),
    ("password_reset", "If you forget your portal password, use the account recovery page.", {"topic": "IT"}),
]
collection.upsert(
    ids=[r[0] for r in records],
    documents=[r[1] for r in records],
    metadatas=[r[2] for r in records],
)
print("Stored records:", collection.count())

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 52.8MiB/s]


Stored records: 5


## Search using natural language


In [3]:
result = collection.query(
    query_texts=["I cannot sign into my student account"],
    n_results=3,
)
for doc, meta, distance in zip(
    result["documents"][0], result["metadatas"][0], result["distances"][0]
):
    print(f"distance={distance:.4f} | {meta['topic']} | {doc}")

distance=0.9594 | IT | Connect to the student Wi-Fi using your campus credentials.
distance=1.1113 | academics | Bring your student identity card to every examination.
distance=1.1518 | IT | If you forget your portal password, use the account recovery page.


## Metadata filtering


In [4]:
filtered = collection.query(
    query_texts=["Where can I play badminton?"],
    where={"topic": "sports"},
    n_results=2,
)
print(filtered["documents"][0])

['The sports center has badminton courts and a fitness room.']


## Inspect, update, and delete


In [5]:
print("IDs:", collection.get()["ids"])
collection.update(
    ids=["library_hours"],
    documents=["The campus library operates from 7:30 AM to 9:30 PM on weekdays."],
    metadatas=[{"topic": "facilities", "verified": True}],
)
print("Updated:", collection.get(ids=["library_hours"])["documents"])

# Demonstrate deletion on a disposable sample record.
collection.upsert(ids=["temporary_notice"], documents=["Temporary maintenance notice."],
                  metadatas=[{"topic": "temporary"}])
collection.delete(ids=["temporary_notice"])
print("Temporary record remaining:", collection.get(ids=["temporary_notice"])["ids"])

IDs: ['library_hours', 'wifi_help', 'exam_rules', 'sports_center', 'password_reset']
Updated: ['The campus library operates from 7:30 AM to 9:30 PM on weekdays.']
Temporary record remaining: []
